# Judge system in Multitool Agentic RAG

In [2]:
# import document : 
import warnings 
warnings.filterwarnings("ignore")
from langchain_community.document_loaders import PyPDFLoader 
loader = PyPDFLoader("economics_research_reference.pdf")
pages = loader.load()

In [3]:
# split the document : 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
spliter = RecursiveCharacterTextSplitter(chunk_size=1400 , chunk_overlap=120)
texts = spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]

# Create Vector Database 
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding = SentenceTransformerEmbeddingFunction()

client = chromadb.PersistentClient(path="./Economics_DB")

collection = client.get_or_create_collection(name="Evelute",embedding_function=embedding)

if collection.count()==0:
    collection.add(
        documents=chunks , 
        ids=[str(i) for i in range(len(chunks))],
        metadatas=metadata
    )

collection.count()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6521.45it/s]


21

In [4]:
# Import tool 
from langchain_core.tools import tool 
from langchain_community.tools import DuckDuckGoSearchRun 
import ast 

# create calculator tool 
@tool 
def calculator(exec:str):
    """ Provided Anytypes of arithmetic operations calculate by this Tool """ 
    try :
        return str(ast.literal_eval(exec))
    except Exception as e :
        return str(e)
# Weather tool 
import requests
@tool 
def Weather(location:str):
    """ Just provide the weather of user provided location """
    try:
        weather = requests.get(f"https://wttr.in/{location}?format=3")
        return weather.text
    except Exception as e: 
        return str(e)

# create local document tool 
@tool 
def retrive(query:str):
    """ Provided user query search for answer on the local document then transfer to the websearch [if it cant find out any related content]""" 
    result = collection.query(query_texts=[query],n_results=3)
    dis = result['distances'][0]
    document = result['documents'][0]
    thresold = 0.9
    goo_chun=[docs for i , docs in zip(dis,document) if i<thresold]

    if goo_chun:
        return "\n\n".join(goo_chun)
    search = DuckDuckGoSearchRun()
    result = search.run(query)
    return result  
@tool 
def file_read(filename:str):
    """ Read the user given file what is the context of it """
    try :
        with open (file=filename) as file:
            read = file.read()
            return f"file content is : \n {read}"
    except Exception as e :
        return str(e)

import os 
@tool 
def file_write(filename :str , content:str):
    """ write the content in user given file name """
    os.makedirs(os.path.dirname(os.path.abspath(filename)),exist_ok=True)
    with open (file=filename) as f : 
        write  = f.write(content)
        return f"write all the content on the user given file : {write}"

tools = [calculator,retrive,Weather,file_read,file_write]
tool_name = {t.name:t for t in tools}
tool_name 

{'calculator': StructuredTool(name='calculator', description='Provided Anytypes of arithmetic operations calculate by this Tool', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x123d86dd0>),
 'retrive': StructuredTool(name='retrive', description='Provided user query search for answer on the local document then transfer to the websearch [if it cant find out any related content]', args_schema=<class 'langchain_core.utils.pydantic.retrive'>, func=<function retrive at 0x123d86d40>),
 'Weather': StructuredTool(name='Weather', description='Just provide the weather of user provided location', args_schema=<class 'langchain_core.utils.pydantic.Weather'>, func=<function Weather at 0x123d86b90>),
 'file_read': StructuredTool(name='file_read', description='Read the user given file what is the context of it', args_schema=<class 'langchain_core.utils.pydantic.file_read'>, func=<function file_read at 0x123d864d0>),
 'file_write': StructuredTool(name='fil

In [5]:
# import llm 
from langchain_groq import ChatGroq 
import os 
from dotenv import load_dotenv 
load_dotenv()
key=os.getenv("GROQ_API_KEY")
chat = ChatGroq(model="llama-3.1-8b-instant")
LLM_bind = chat.bind_tools(tools) # LLM tool blind use 

In [6]:
!uv pip install -U ddgs

Resolved 18 packages in 384ms                                        
Checked 18 packages in 1ms


In [7]:
from langchain_core.messages import ToolMessage

# Multi tool agentic function 
def tool_fun(question):
    messages = [{"role": "user", "content": question}]
    response = LLM_bind.invoke(messages)

    if not response.tool_calls:
        return response.content
    retrive_content=[]
    messages.append(response)

    for call in response.tool_calls:
        tool = tool_name[call['name']]
        result = tool.invoke(call['args'])
        retrive_content.append(result)
        messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

    final = LLM_bind.invoke(messages)
    return final.content ,  retrive_content # need both bec to create a judge system what model give output and what retrive come back to 


query = "what is the current weather in kolkata ??"
generated_answer = tool_fun(query)
response = generated_answer[-1]
print(response)

['Kolkata: 🌤️  +29°C\n']


In [9]:
from pydantic import BaseModel, Field
from typing import Literal 


# Create it for the Judge system 
class Build(BaseModel):
    reasoning: str = Field(description="A one-line explanation of why this verdict was chosen.")
    verdict: Literal["fully_grounded", "partial_grounded", "hallucinated"]

def judge(query: str, context :str , answer: str):
    # Prompt for strict judge and prevent halucinate answer 
    prompt = f"""You are a strict RAG judge. Judge the generated answer based ONLY on the provided query.

Query: {query}
Context: {context}
Answer: {answer}

Instructions for Verdict:
- fully_grounded: answer is entirely supported by the context.
- partial_grounded: answer relates to the context but adds unsupported detail.
- hallucinated: answer comes from memory, not the context (or context is empty/None).

if user asking any question's answers find out in the local document then direct give answer and if dont then directly say "NOT RELATED CONTENT"

"""

    evaluator = chat.with_structured_output(Build)
    return evaluator.invoke(prompt)

queries = [
    "what is Demand ?",
    "what is 12*8?", 
    "what is the largest country in the world right now ?"
    "what is the current weather now in barrackpore ?"
]
for qustion in queries:
    content , generated_answer = tool_fun(qustion)   
    content = content 
    generated_answer=generated_answer
    print(f'the agent output : {content}')
    evelute = judge(query=qustion , context=content, answer=generated_answer)
    print(f'the reaseon : {evelute.reasoning}')
    print(f'the vedict : {evelute.verdict}')

the agent output : 
the reaseon : The answer is related to economics and the provided text is about microeconomics, but it does not directly define 'Demand'.
the vedict : partial_grounded
the agent output : It seems like the calculator function does not support multiplication. Let me try again with a different operation.


the reaseon : The answer is a message indicating a syntax error or a problem with the node or string, but it does not provide the result of the operation.
the vedict : hallucinated
the agent output : 
the reaseon : The generated answer does not relate to the query about the largest country in the world and the current weather in Barrackpore, but instead provides unrelated information about economics and macroeconomics.
the vedict : hallucinated
